# IABR-Net — Full Test Suite (all experiments from the multi-agent research report)

One notebook that builds, trains, and tests **IABR-Net** (Impairment-Adaptive Beamspace
Residual-correction Network, Candidate B of `ResearchState/FINAL_RESEARCH_REPORT.md`) and every
baseline it must be compared against, then writes every number, table, figure and checkpoint to
`outputs/`.

**Run order on a new machine:** set `SMOKE_TEST = True` in the config cell and *Run All* first
(a few minutes) to confirm the environment works end to end, then set it back to `False` and run
the full suite. Every experiment caches its result in `outputs/results/<name>.json`, and every
training run checkpoints, so an interrupted run resumes where it stopped when you *Run All* again.

| Section | Experiments |
|---|---|
| Phase 0 | **E0a** gain/phase-error injection + physics unit tests, **E0b** real params / FLOPs / memory |
| Reproducibility | **V0** teacher reproduces the registry number on the frozen bank, **V1** new-bank generator fidelity |
| Training | IABR-Net × 3 seeds, **E1** capacity-matched control, **E2** dense front-end, **Abl-1** no IABC-v2, **Abl-3** no SE, **Abl-6** 12-run loss-weight sweep |
| Accuracy | **E3** in-distribution SNR sweep (full 8000-sample frozen bank), **Abl-2** refinement / end-fire P95 |
| Robustness | **E4** phase error, **E5** gain error, **E6** OOD phase, **E7** path count, **E8** angular separation, SNR tails, **E10** nuisance path |
| Statistics | **E11** multi-seed variance |
| Efficiency | **E9** NN-only and end-to-end latency, FLOPs, memory, model size for every model |

Implementation decisions that the research report left open are marked **[ASSUMPTION]** in the
cells and listed in the README.

In [ ]:
# ---------------------------------------------------------------- 1. Setup
import os, sys, json, time, math, traceback, hashlib, subprocess, platform, datetime
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')

# >>> Run mode. SMOKE_TEST=True: tiny sizes, checks the whole pipeline in minutes, writes to outputs_smoke/.
# >>> BANKS_ONLY=True: only run the Phase-0 unit tests and build the data banks.
SMOKE_TEST = os.environ.get('IABR_SMOKE', '0') == '1'
BANKS_ONLY = os.environ.get('IABR_BANKS_ONLY', '0') == '1'

import numpy as np
import scipy
import scipy.ndimage
from scipy.optimize import linear_sum_assignment
import matplotlib
if 'ipykernel' not in sys.modules:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python-headless'], check=True)
    import cv2

import tensorflow as tf
from tensorflow.keras import layers as KL
from tensorflow.keras.layers import Conv2D, Input, BatchNormalization, Activation, Add, Conv2DTranspose, Layer
from tensorflow.keras.models import Model

tf.get_logger().setLevel('ERROR')
T_START = time.time()

GPUS = tf.config.list_physical_devices('GPU')
for g in GPUS:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except RuntimeError:
        pass
DEVICE = 'GPU' if GPUS else 'CPU'


def find_root():
    here = os.path.abspath(os.getcwd())
    for _ in range(4):
        if os.path.exists(os.path.join(here, 'data', 'frozen_banks', 'eval_bank.npz')):
            return here
        here = os.path.dirname(here)
    raise FileNotFoundError('Could not find the package root (a folder containing data/frozen_banks/eval_bank.npz). '
                            'Open this notebook from inside the IABR_Net_TestSuite folder.')


ROOT = find_root()
OUT = os.path.join(ROOT, 'outputs_smoke' if SMOKE_TEST else 'outputs')   # smoke results never mix with the real run
DIRS = {k: os.path.join(OUT, k) for k in ['results', 'figures', 'checkpoints', 'logs', 'cache', 'tables']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)
LOG_FILE = os.path.join(DIRS['logs'], 'run_log.txt')


def log(*args):
    msg = ' '.join(str(a) for a in args)
    stamp = f'[{(time.time() - T_START) / 60:7.1f} min] '
    print(stamp + msg, flush=True)
    with open(LOG_FILE, 'a', encoding='utf-8') as f:
        f.write(stamp + msg + '\n')


log('=' * 70)
log('IABR-Net test suite start', datetime.datetime.now().isoformat(timespec='seconds'))
log(f'TF {tf.__version__} | numpy {np.__version__} | device: {DEVICE} {GPUS}')
log(f'ROOT = {ROOT}')
if not GPUS:
    log('WARNING: no GPU visible to TensorFlow. Full training on CPU will be very slow. '
        'TensorFlow >= 2.11 has no native-Windows GPU support; use Linux/WSL2 with CUDA, or macOS + tensorflow-metal.')

In [ ]:
# ---------------------------------------------------------------- 2. Configuration (SMOKE_TEST / BANKS_ONLY are set in cell 1)
CFG = dict(
    full_steps=20000,          # step-count-matched to the project's 4-way screening (report section 20)
    sweep_steps=5000,          # Abl-6 loss-weight sweep runs are shorter [ASSUMPTION]
    batch=256,
    lr_max=1e-3, lr_min=1e-5, warmup=500,   # [ASSUMPTION] the report left optimizer/schedule undecided
    clipnorm=5.0,
    seeds=[0, 1, 2],           # E11 multi-seed
    ckpt_every=2000,
    log_every=200,
    lambdas=dict(heat=1.0, off=1.0, pair=0.5, snr=0.1),   # report section 19 starting weights
    train_phase_deg_max=5.0,   # [ASSUMPTION] training impairment ranges: phase U(0,5) deg, gain U(0,2) dB per sample
    train_gain_db_max=2.0,
    train_L=(1, 9),            # same as the original training generator (L in 1..9)
    train_snr=(-15, 24),       # same as the original training generator
    n_robust=500,              # samples per robustness condition
    robust_snrs=[0, 15],
    e3_n_per_snr=None,         # None = full frozen bank (1000 / SNR)
    nuisance_scenes=200,
    sweep_l3=[0.0, 0.1, 0.5, 1.0], sweep_l4=[0.0, 0.1, 0.3],
    sweep_eval_n_per_snr=200,
    dft_ndft=512,              # classical DFT-SIC grid [ASSUMPTION] (1024 in the sanity notebook; 512 keeps eval time sane)
    latency_batches=[1, 8, 32, 128],
    latency_iters=30, latency_e2e_samples=100,
    eval_batch=64,
    data_workers=4,
    k_slots=10,
    heat_sigma_cells=0.8,
)
if SMOKE_TEST:
    CFG.update(full_steps=40, sweep_steps=10, batch=16, warmup=5, ckpt_every=20, log_every=10,
               seeds=[0, 1], n_robust=6, e3_n_per_snr=4, nuisance_scenes=3,
               sweep_l3=[0.0, 0.5], sweep_l4=[0.1], sweep_eval_n_per_snr=3,
               dft_ndft=128, latency_batches=[1, 4], latency_iters=3, latency_e2e_samples=3,
               eval_batch=8, data_workers=1)
    OUT_BANKS = os.path.join(OUT, 'smoke_banks')
else:
    OUT_BANKS = os.path.join(ROOT, 'data', 'generated_banks')
os.makedirs(OUT_BANKS, exist_ok=True)

P = Q = NT = NR = 16
SIGMA_GT = 0.07
M_GT = 256
log('SMOKE_TEST =', SMOKE_TEST, '| BANKS_ONLY =', BANKS_ONLY)
log('CFG =', json.dumps(CFG))
with open(os.path.join(DIRS['logs'], 'config.json'), 'w') as f:
    json.dump(dict(CFG=CFG, SMOKE_TEST=SMOKE_TEST, device=DEVICE, tf=tf.__version__,
                   python=sys.version, platform=platform.platform()), f, indent=2)

In [ ]:
# ---------------------------------------------------------------- 3. Experiment runner (cache + resume + keep going on errors)
STATUS_PATH = os.path.join(DIRS['results'], 'status.json')
STATUS = json.load(open(STATUS_PATH)) if os.path.exists(STATUS_PATH) else {}


def _to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): _to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_to_jsonable(v) for v in o]
    if isinstance(o, (np.floating,)):
        return None if not np.isfinite(o) else float(o)
    if isinstance(o, float):
        return None if not math.isfinite(o) else o
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    if isinstance(o, np.ndarray):
        return _to_jsonable(o.tolist())
    return o


def save_json(obj, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(_to_jsonable(obj), f, indent=2)


def run_exp(name, fn, needs=None):
    """Run one experiment. Result cached in outputs/results/<name>.json -- delete that file to re-run.
    A failure is logged to outputs/logs/errors.log and the suite keeps going."""
    path = os.path.join(DIRS['results'], f'{name}.json')
    if BANKS_ONLY and not name.startswith(('E0a', 'BANKS')):
        return None
    if os.path.exists(path):
        log(f'[{name}] cached -> {os.path.relpath(path, ROOT)}')
        return json.load(open(path, encoding='utf-8'))
    for dep in (needs or []):
        if not str(STATUS.get(dep, '')).startswith('done'):
            log(f'[{name}] SKIPPED -- depends on {dep} which is {STATUS.get(dep, "not run")}')
            STATUS[name] = f'skipped (needs {dep})'
            save_json(STATUS, STATUS_PATH)
            return None
    log(f'[{name}] start')
    t0 = time.time()
    try:
        res = fn()
        res = dict(res or {})
        res['_elapsed_min'] = (time.time() - t0) / 60
        res['_smoke_test'] = SMOKE_TEST
        save_json(res, path)
        STATUS[name] = f'done ({res["_elapsed_min"]:.1f} min)'
        log(f'[{name}] done in {res["_elapsed_min"]:.1f} min')
        save_json(STATUS, STATUS_PATH)
        return json.load(open(path, encoding='utf-8'))
    except Exception as e:
        tb = traceback.format_exc()
        with open(os.path.join(DIRS['logs'], 'errors.log'), 'a', encoding='utf-8') as f:
            f.write(f'\n===== {name} =====\n{tb}\n')
        STATUS[name] = f'FAILED: {type(e).__name__}: {e}'
        save_json(STATUS, STATUS_PATH)
        log(f'[{name}] FAILED: {type(e).__name__}: {e} (traceback in outputs/logs/errors.log)')
        return None

In [ ]:
# ---------------------------------------------------------------- 4. Original project code (evaluator, teacher, generator)
from pathlib import Path


def find_file(pattern):
    for p in Path(ROOT).rglob(pattern):
        if p.is_file() and 'outputs' not in p.parts:
            return str(p)
    raise FileNotFoundError(pattern)


DL_DOA_DIR = os.path.dirname(os.path.dirname(find_file('tvt_models.py')))
GEN_FILE = find_file('dldoa_dataset_generation.py')
sys.path.insert(0, DL_DOA_DIR)
sys.path.insert(0, os.path.dirname(GEN_FILE))
from src.tvt_models import Resnet
from src.TVT_Blob_Inference import (get_blob_detector, get_blob_peaks, peaks_to_angles,
                                    prepare_for_metric, get_ang_difference, filter_angles)
import dldoa_dataset_generation as DG

TEACHER_W = find_file('inf_model_007_256_resnet.h5')
STUDENT_W = find_file('student_r8_magnitude_lambda0.5.weights.h5')
FNO_W = find_file('FNO.weights.h5')
EVAL_BANK = os.path.join(ROOT, 'data', 'frozen_banks', 'eval_bank.npz')
log('evaluator + generator imported from', DL_DOA_DIR)
log('teacher:', os.path.relpath(TEACHER_W, ROOT), '| student:', os.path.relpath(STUDENT_W, ROOT),
    '| FNO:', os.path.relpath(FNO_W, ROOT))

## Physics and data

The observation is `Y = Wᴴ H F + Z` with DFT codebooks `F`, `W` (16×16, square), so `Y` is already
beamspace. Hardware impairment is a per-antenna complex error on every codebook column:
`F = D_t F_ideal`, `W = D_r W_ideal`, with `D = diag(g·e^{jε})`.

- **Phase error** `ε ~ U(−δmax, δmax)`, exactly as in the original generator.
- **Gain error (new, Phase-0 code)** `20·log10(g) ~ U(−γmax, γmax)` dB. **[ASSUMPTION]** This uses the
  same symmetric-uniform convention as the phase error, because the report names the gain levels but
  not their distribution.

Because the codebooks are unitary, `W_ideal · Y · F_idealᴴ = D_rᴴ H D_t + noise`. So the per-element
error is exactly two diagonal matrices in the antenna domain, which is where IABR-Net's IABC-v2
module corrects it (report Addendum A1).

Banks are stored compactly as the 16×16 `Y`. The 64×64 input of the heatmap baselines is rebuilt
with the exact `scipy.ndimage.zoom(order=0)` index map the original generator uses, so the
baselines see bit-identical inputs (checked in E0a).

In [ ]:
# ---------------------------------------------------------------- 5. Vectorized simulator with phase + gain impairments
F_IDEAL = DG.beamforming_vector_generation_P(P, NT)          # (nt, P), no impairment, exact original code
W_IDEAL = DG.beamforming_vector_generation_Q(Q, NR)          # (nr, Q)
COS_T = (1 / np.pi) * np.angle(np.exp(1j * (2 * np.pi / P) * np.arange(P)))    # cos(phi) of Tx beam p
COS_R = (1 / np.pi) * np.angle(np.exp(-1j * (2 * np.pi / Q) * np.arange(Q)))   # cos(psi) of Rx beam q

UP_SRC = scipy.ndimage.zoom(np.arange(P * Q, dtype=np.float64).reshape(Q, P), 4, order=0).astype(np.int64).ravel()
DOWN_IDX = np.array([int(np.argmax(UP_SRC == s)) for s in range(P * Q)])


def upsample64(y16):
    """(N,16,16,2) -> (N,64,64,2), identical to scipy.ndimage.zoom(order=0, zoom=4) of the original generator."""
    n = y16.shape[0]
    return y16.reshape(n, P * Q, 2)[:, UP_SRC, :].reshape(n, 64, 64, 2)


def downsample16(y64):
    n = y64.shape[0]
    return y64.reshape(n, 64 * 64, 2)[:, DOWN_IDX, :].reshape(n, Q, P, 2)


def sample_paths(L_arr, rng, sep=np.pi / 6, lmax=None):
    """Angles (AoA psi, AoD phi) with min separation sep, alpha ~ CN(0,1/L) sorted strongest first."""
    B = len(L_arr)
    lmax = lmax or int(max(L_arr))
    psi = np.full((B, lmax), np.nan); phi = np.full((B, lmax), np.nan)
    alpha = np.zeros((B, lmax), complex)
    for b, L in enumerate(L_arr):
        L = int(L)
        pts = DG.generate_points(L, sep, rng=rng)           # list of (AoD, AoA), exact original sampler
        a = np.sqrt(1 / L) * (rng.standard_normal(L) + 1j * rng.standard_normal(L)) / np.sqrt(2)
        a = a[np.argsort(-np.abs(a))]
        phi[b, :L] = [p[0] for p in pts]; psi[b, :L] = [p[1] for p in pts]
        alpha[b, :L] = a
    return psi, phi, alpha


def simulate(psi, phi, alpha, snr_db, rng, phase_deg_max=0.0, gain_db_max=0.0, want_clean=False, noise=None):
    """Batch of observations. psi/phi/alpha: (B,Lmax), padded paths have alpha=0 (angles may be nan).
    Returns Y (B,Q,P) complex and, if want_clean, Y_clean = W_idealᴴ H F_ideal + the SAME noise."""
    B, Lm = alpha.shape
    ps = np.nan_to_num(psi); ph = np.nan_to_num(phi)
    k_r = np.arange(NR); k_t = np.arange(NT)
    a_r = np.exp(-1j * np.pi * np.cos(ps)[..., None] * k_r) / np.sqrt(NR)       # (B,Lm,nr)
    a_t = np.exp(-1j * np.pi * np.cos(ph)[..., None] * k_t) / np.sqrt(NT)       # (B,Lm,nt)
    H = np.sqrt(NT * NR) * np.einsum('bl,bln,blm->bnm', alpha, a_r, np.conj(a_t))
    pdm = np.broadcast_to(np.asarray(phase_deg_max, float), (B,))
    gdm = np.broadcast_to(np.asarray(gain_db_max, float), (B,))
    eps_r = rng.uniform(-1, 1, (B, NR)) * np.deg2rad(pdm)[:, None]
    eps_t = rng.uniform(-1, 1, (B, NT)) * np.deg2rad(pdm)[:, None]
    g_r = 10 ** (rng.uniform(-1, 1, (B, NR)) * gdm[:, None] / 20)
    g_t = 10 ** (rng.uniform(-1, 1, (B, NT)) * gdm[:, None] / 20)
    D_r = g_r * np.exp(1j * eps_r); D_t = g_t * np.exp(1j * eps_t)
    W = D_r[:, :, None] * W_IDEAL[None]                                           # (B,nr,Q)
    F = D_t[:, :, None] * F_IDEAL[None]                                           # (B,nt,P)
    G = np.einsum('bnq,bnm,bmp->bqp', np.conj(W), H, F)
    if noise is None:
        var = 10 ** (-np.broadcast_to(np.asarray(snr_db, float), (B,)) / 10)
        s = np.sqrt(var / 2)[:, None, None]
        noise = s * (rng.standard_normal((B, Q, P)) + 1j * rng.standard_normal((B, Q, P)))
    Y = G + noise
    out = dict(Y=Y, D_r=D_r, D_t=D_t, H=H)
    if want_clean:
        Gc = np.einsum('nq,bnm,mp->bqp', np.conj(W_IDEAL), H, F_IDEAL)
        out['Y_clean'] = Gc + noise
    return out


def to_ri(Y):
    return np.stack([Y.real, Y.imag], axis=-1).astype(np.float32)


def wrap2pi(x):
    return np.mod(x, 2 * np.pi)


def angles_to_cells(psi, phi):
    """Continuous beamspace coordinates (q*, p*) in [0,16): rows = AoA, cols = AoD (same axes as the GT image)."""
    q = Q * wrap2pi(-np.pi * np.cos(psi)) / (2 * np.pi)
    p = P * wrap2pi(np.pi * np.cos(phi)) / (2 * np.pi)
    return q, p


def cells_to_angles(q, p):
    """Inverse of angles_to_cells, same wrap convention as TVT_Blob_Inference.peaks_to_angles."""
    wq = wrap2pi(2 * np.pi * np.asarray(q) / Q); wp = wrap2pi(2 * np.pi * np.asarray(p) / P)
    wq = np.where(wq > np.pi, wq - 2 * np.pi, wq); wp = np.where(wp > np.pi, wp - 2 * np.pi, wp)
    return np.arccos(np.clip(-wq / np.pi, -1, 1)), np.arccos(np.clip(wp / np.pi, -1, 1))


log('simulator ready | upsample map: %d distinct sources' % len(np.unique(UP_SRC)))

In [ ]:
# ---------------------------------------------------------------- 6. E0a -- Phase-0 unit tests (gain injection + physics)
def e0a():
    checks = {}
    rng = np.random.default_rng(123)
    # 1. codebooks are unitary -> inverse codebook transform is exact
    checks['W_ideal unitary'] = float(np.abs(W_IDEAL.conj().T @ W_IDEAL - np.eye(Q)).max())
    checks['F_ideal unitary'] = float(np.abs(F_IDEAL.conj().T @ F_IDEAL - np.eye(P)).max())
    # 2. vectorized simulator == original generator functions (no impairment, same angles/gains/noise)
    psi, phi, alpha = sample_paths([3, 5], rng)
    noise = 0.1 * (rng.standard_normal((2, Q, P)) + 1j * rng.standard_normal((2, Q, P)))
    sim = simulate(psi, phi, alpha, 10, rng, noise=noise)
    err = 0.0
    for b, L in enumerate([3, 5]):
        H0 = DG.generate_channel_v2(NR, NT, np.hstack([phi[b, :L], psi[b, :L]]), alpha[b, :L])
        Y0 = (W_IDEAL.conj().T @ H0) @ F_IDEAL + noise[b]
        err = max(err, float(np.abs(Y0 - sim['Y'][b]).max()))
    checks['simulator == original generate_channel_v2 path'] = err
    # 3. gain + phase injection: codebook columns carry exactly g*e^{j eps} per antenna
    sim = simulate(psi[:1], phi[:1], alpha[:1], 10, rng, phase_deg_max=5, gain_db_max=2, want_clean=True)
    D_r, D_t = sim['D_r'][0], sim['D_t'][0]
    checks['gain within +-2 dB'] = float(np.abs(20 * np.log10(np.abs(np.r_[D_r, D_t]))).max())
    checks['phase within +-5 deg'] = float(np.degrees(np.abs(np.angle(np.r_[D_r, D_t]))).max())
    Wimp = D_r[:, None] * W_IDEAL; Fimp = D_t[:, None] * F_IDEAL
    Ynf = (Wimp.conj().T @ sim['H'][0]) @ Fimp
    Ht = W_IDEAL @ Ynf @ F_IDEAL.conj().T
    checks['inverse codebook recovers D_r^H H D_t'] = float(np.abs(Ht - np.conj(D_r)[:, None] * sim['H'][0] * D_t[None, :]).max())
    # 4. original phase-error code path agrees (per-antenna, same for all beams)
    np.random.seed(7)
    Fp = DG.beamforming_vector_generation_P(P, NT, error_deg=5)
    ratio = Fp / F_IDEAL
    checks['original phase error is per-antenna (same across beams)'] = float(np.abs(ratio - ratio[:, :1]).max())
    # 5. compact 16x16 storage is exactly invertible on the frozen bank (and it is the zoom-4 P=16 case)
    fb = np.load(EVAL_BANK)
    d = fb['data'][:64]
    checks['frozen bank: upsample(downsample(x)) == x'] = float(np.abs(upsample64(downsample16(d)) - d).max())
    checks['frozen bank P values'] = sorted(set(fb['meta'][:, 2].astype(int).tolist()))
    # 6. single noiseless path: |Y| peak lands in the cell predicted by angles_to_cells, and cells<->angles round-trips
    worst_cell, worst_rt = 0, 0.0
    for _ in range(200):
        ps, ph, al = sample_paths([1], rng)
        Y = simulate(ps, ph, al, 300, rng)['Y'][0]
        qi, pi_ = np.unravel_index(np.argmax(np.abs(Y)), Y.shape)
        qc, pc = angles_to_cells(ps[0, 0], ph[0, 0])
        dq = min(abs(qi - qc), Q - abs(qi - qc)); dp = min(abs(pi_ - pc), P - abs(pi_ - pc))
        worst_cell = max(worst_cell, dq, dp)
        a1, a2 = cells_to_angles(qc, pc)
        worst_rt = max(worst_rt, abs(np.cos(a1) - np.cos(ps[0, 0])), abs(np.cos(a2) - np.cos(ph[0, 0])))
    checks['|Y| argmax within 0.5 cell of predicted cell (200 single paths)'] = float(worst_cell)
    checks['cells->angles round trip (max |cos err|)'] = float(worst_rt)
    tol = {'W_ideal unitary': 1e-10, 'F_ideal unitary': 1e-10, 'simulator == original generate_channel_v2 path': 1e-10,
           'gain within +-2 dB': 2.0 + 1e-9, 'phase within +-5 deg': 5.0 + 1e-9,
           'inverse codebook recovers D_r^H H D_t': 1e-9,
           'original phase error is per-antenna (same across beams)': 1e-12,
           'frozen bank: upsample(downsample(x)) == x': 0.0,
           '|Y| argmax within 0.5 cell of predicted cell (200 single paths)': 0.5 + 1e-9,
           'cells->angles round trip (max |cos err|)': 1e-9}
    passed = {k: (v <= tol[k]) for k, v in checks.items() if k in tol}
    passed['frozen bank P values'] = checks['frozen bank P values'] == [16]
    for k in checks:
        log(f'  E0a  {"PASS" if passed.get(k) else "FAIL"}  {k}: {checks[k]}')
    assert all(passed.values()), 'E0a physics checks failed -- do not trust anything downstream'
    return dict(checks=checks, passed=passed, all_passed=True)


R_E0A = run_exp('E0a_phase0_unit_tests', e0a)

## Data banks

`data/frozen_banks/eval_bank.npz` is used unchanged for **E3**. It is the project's fixed
8000-sample bank, and every registry number comes from it. All new conditions are generated
deterministically with fixed seeds and cached in `data/generated_banks/`. The package ships them
pre-generated; delete a file to regenerate it.

**[ASSUMPTION]** Robustness conditions are evaluated at SNR ∈ {0, 15} dB, the same hard and moderate
pair used in the project's Notebook 4. Each condition has `n_robust` samples.

**Correction to the report:** the original training generator draws L from {1..9}. So L = 7 and
L = 8 are *in-distribution* for the teacher and for IABR-Net. Only L = 10 is OOD.

In [ ]:
# ---------------------------------------------------------------- 7. Build / load all banks
def bank_path(name):
    return os.path.join(OUT_BANKS, f'{name}.npz')


def make_standard_bank(name, n, L, snr, seed, phase_deg=0.0, gain_db=0.0):
    if os.path.exists(bank_path(name)):
        return
    rng = np.random.default_rng(seed)
    psi, phi, alpha = sample_paths([L] * n, rng)
    Y = simulate(psi, phi, alpha, np.full(n, snr), rng, phase_deg_max=phase_deg, gain_db_max=gain_db)['Y']
    np.savez_compressed(bank_path(name), Y16=to_ri(Y), psi=psi.astype(np.float32), phi=phi.astype(np.float32),
                        L=np.full(n, L, np.int32), snr=np.full(n, snr, np.float32),
                        phase_deg=np.float32(phase_deg), gain_db=np.float32(gain_db), seed=np.int64(seed))


def make_separation_bank(name, n, sep_deg, snr, seed):
    """L=2, both AoA and AoD of the second path offset by sep_deg from the first; angles kept in [20,160] deg."""
    if os.path.exists(bank_path(name)):
        return
    rng = np.random.default_rng(seed)
    d = np.deg2rad(sep_deg)
    psi1 = rng.uniform(np.deg2rad(20), np.deg2rad(160) - d, n)
    phi1 = rng.uniform(np.deg2rad(20), np.deg2rad(160) - d, n)
    psi = np.stack([psi1, psi1 + d], 1); phi = np.stack([phi1, phi1 + d], 1)
    alpha = np.sqrt(1 / 2) * (rng.standard_normal((n, 2)) + 1j * rng.standard_normal((n, 2))) / np.sqrt(2)
    order = np.argsort(-np.abs(alpha), axis=1)
    alpha = np.take_along_axis(alpha, order, 1); psi = np.take_along_axis(psi, order, 1); phi = np.take_along_axis(phi, order, 1)
    Y = simulate(psi, phi, alpha, np.full(n, snr), rng)['Y']
    np.savez_compressed(bank_path(name), Y16=to_ri(Y), psi=psi.astype(np.float32), phi=phi.astype(np.float32),
                        L=np.full(n, 2, np.int32), snr=np.full(n, snr, np.float32), sep_deg=np.float32(sep_deg),
                        seed=np.int64(seed))


def make_nuisance_bank(name, n_scenes, snr, power_db, seed):
    """Notebook-4 protocol: 3 principal paths (L=3 gain law) + 1 nuisance at power_db relative to the
    principal total; same scenes and noise across power levels (paired). feat index 3 = nuisance."""
    if os.path.exists(bank_path(name)):
        return
    rng = np.random.default_rng(seed)           # scene sequence depends only on (seed, snr) -> paired across power
    Ys, psis, phis = [], [], []
    for _ in range(n_scenes):
        pts = DG.generate_points(4, np.pi / 6, rng=rng)
        ph = np.array([p[0] for p in pts]); ps = np.array([p[1] for p in pts])
        a_p = (np.sqrt(1 / 3) / np.sqrt(2)) * (rng.standard_normal(3) + 1j * rng.standard_normal(3))
        noise_seed = int(rng.integers(0, 2 ** 31 - 1))
        mag = np.sqrt(10 ** (power_db / 10) * np.sum(np.abs(a_p) ** 2))
        a_n = mag * np.exp(1j * np.random.default_rng(noise_seed + 777).uniform(0, 2 * np.pi))
        al = np.concatenate([a_p, [a_n]])[None]
        rn = np.random.default_rng(noise_seed)
        s = np.sqrt(10 ** (-snr / 10) / 2)
        Z = (s * (rn.standard_normal((Q, P)) + 1j * rn.standard_normal((Q, P))))[None]
        Ys.append(simulate(ps[None], ph[None], al, snr, rng, noise=Z)['Y'][0]); psis.append(ps); phis.append(ph)
    np.savez_compressed(bank_path(name), Y16=to_ri(np.array(Ys)), psi=np.array(psis, np.float32),
                        phi=np.array(phis, np.float32), L=np.full(n_scenes, 4, np.int32),
                        snr=np.full(n_scenes, snr, np.float32), power_db=np.float32(power_db), seed=np.int64(seed))


N_R = CFG['n_robust']
BANK_SPECS = {}
for snr in CFG['robust_snrs']:
    for d in [0, 1, 2, 5]:
        BANK_SPECS[f'phase_d{d}_snr{snr}'] = ('std', dict(n=N_R, L=3, snr=snr, seed=1000 + 10 * d + snr, phase_deg=d))
    for g in [0.5, 1, 2, 4]:
        BANK_SPECS[f'gain_g{g}_snr{snr}'] = ('std', dict(n=N_R, L=3, snr=snr, seed=2000 + int(10 * g) + snr, gain_db=g))
    for d in [10, 15]:
        BANK_SPECS[f'oodphase_d{d}_snr{snr}'] = ('std', dict(n=N_R, L=3, snr=snr, seed=3000 + d + snr, phase_deg=d))
    for L in [1, 2, 4, 5, 6, 7, 8, 10]:
        BANK_SPECS[f'L{L}_snr{snr}'] = ('std', dict(n=N_R, L=L, snr=snr, seed=4000 + 10 * L + snr))
    for pw in [-20, -10, 0]:
        BANK_SPECS[f'nuis_p{pw}_snr{snr}'] = ('nuis', dict(n_scenes=CFG['nuisance_scenes'], snr=snr, power_db=pw,
                                                            seed=1000 + snr))
for s in [30, 20, 10, 5, 3, 2, 1]:
    BANK_SPECS[f'sep_{s}deg_snr15'] = ('sep', dict(n=N_R, sep_deg=s, snr=15, seed=5000 + s))
for snr in [-25, -20, 30, 35]:
    BANK_SPECS[f'tail_snr{snr}'] = ('std', dict(n=N_R, L=3, snr=snr, seed=6000 + snr))
for snr in [0, 15, 25]:
    BANK_SPECS[f'v1_clean_snr{snr}'] = ('std', dict(n=N_R, L=3, snr=snr, seed=7000 + snr))


def build_banks():
    t0 = time.time()
    for name, (kind, kw) in BANK_SPECS.items():
        if kind == 'std':
            make_standard_bank(name, **kw)
        elif kind == 'sep':
            make_separation_bank(name, **kw)
        else:
            make_nuisance_bank(name, **kw)
    manifest = {}
    for name in BANK_SPECS:
        with open(bank_path(name), 'rb') as f:
            manifest[name] = dict(sha256=hashlib.sha256(f.read()).hexdigest(), spec=BANK_SPECS[name][1])
    save_json(manifest, os.path.join(OUT_BANKS, 'MANIFEST.json'))
    return dict(n_banks=len(BANK_SPECS), dir=os.path.relpath(OUT_BANKS, ROOT), build_min=(time.time() - t0) / 60)


R_BANKS = run_exp('BANKS_build', build_banks, needs=['E0a_phase0_unit_tests'])
_BANK_CACHE = {}


def load_bank(name):
    if name in _BANK_CACHE:
        return _BANK_CACHE[name]
    if name == 'E3_frozen':
        fb = np.load(EVAL_BANK)
        data, feat, meta = fb['data'], fb['feat'], fb['meta']
        if CFG['e3_n_per_snr']:
            idx = np.concatenate([np.arange(i * 1000, i * 1000 + CFG['e3_n_per_snr']) for i in range(8)])
            data, feat, meta = data[idx], feat[idx], meta[idx]
        b = dict(Y16=downsample16(data), psi=feat[:, 0, :], phi=feat[:, 1, :], L=meta[:, 0].astype(int),
                 snr=meta[:, 1].astype(np.float32))
    else:
        z = np.load(bank_path(name))
        b = {k: z[k] for k in z.files}
    _BANK_CACHE[name] = b
    return b


log(f'{len(BANK_SPECS)} generated banks in {os.path.relpath(OUT_BANKS, ROOT)}')

## Models

- **Baselines**, exactly as in `baseline_models/`: the Teacher (original 64-block ResNet), the
  magnitude-pruned Student (r = 8), the FNO screening model, and **classical DFT-SIC**
  (0 learned parameters, same algorithm as `Classical_Baseline_SanityTest.ipynb`).
- **IABR-Net**, as specified in report §16–§19, with the Addendum-A1 correction to stage 0.

IABR-Net stages:
1. **Fixed stage 0:** inverse codebook transform `H̃ = W_ideal · Y · F_idealᴴ` into the antenna domain.
2. **IABC-v2:** per-element shared-weight MLP (10→16→2) with pooled global context (4→8). It predicts
   the per-element phase and log-gain distortion. The correction is applied, then the forward codebook
   maps back to beamspace.
   - **[ASSUMPTION]** The report's "own(mag, phase)" is not defined. It is implemented as the element's
     log-energy ratio, plus its phase residual relative to the dominant path after removing the linear
     trend (a linear phase slope cannot be identified, because it is the same as an angle shift).
     Features are stop-gradient.
3. **Trunk:** 10 residual blocks at C=32 with SE (r=8), held at 16×16.
   - **[ASSUMPTION]** Convolutions use circular padding, because DFT beamspace is periodic.
4. **Heads:** path-token 1×1 conv, coarse 16×16 heatmap, 7×7 crop refinement heads for AoA and AoD
   offsets, and the SE→SNR auxiliary head.
   - **[ASSUMPTION]** Training follows CenterNet-style decoding: focal loss on the heatmap, and offsets
     trained at ground-truth peak cells. At inference, offsets are read at the top-L 3×3 local maxima.

In [ ]:
# ---------------------------------------------------------------- 8. Baseline models
def res_conv_pruned(x, r, out_filters=12):
    skip = x
    x = Conv2D(r, 5, padding='same')(x); x = BatchNormalization()(x); x = Activation('relu')(x)
    x = Conv2D(out_filters, 5, padding='same')(x); x = BatchNormalization()(x)
    x = Add()([x, skip]); x = Activation('relu')(x)
    return x


def build_pruned_resnet(r, n_blocks=64, input_shape=(64, 64, 2), name=None):
    x_in = Input(shape=input_shape)
    x = Conv2DTranspose(12, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(n_blocks):
        x = res_conv_pruned(x, r)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name=name or f'PrunedResNet-r{r}')


class SpectralConv2D(Layer):
    """FNO spectral convolution -- identical to DLDOA_Architecture_FullEval_Local.ipynb so FNO.weights.h5 loads."""
    def __init__(self, out_channels, modes=12, **kwargs):
        super().__init__(**kwargs)
        self.out_channels = out_channels
        self.modes = modes

    def build(self, input_shape):
        in_ch = input_shape[-1]
        self.w_real = self.add_weight(shape=(self.modes, self.modes, in_ch, self.out_channels),
                                      initializer='glorot_uniform', trainable=True, name='w_real')
        self.w_imag = self.add_weight(shape=(self.modes, self.modes, in_ch, self.out_channels),
                                      initializer='glorot_uniform', trainable=True, name='w_imag')

    def call(self, x):
        H, W = x.shape[1], x.shape[2]
        x_ft = tf.transpose(tf.signal.rfft2d(tf.transpose(x, [0, 3, 1, 2])), [0, 2, 3, 1])
        m1 = min(self.modes, H); m2 = min(self.modes, x_ft.shape[2])
        Wc = tf.complex(self.w_real[:m1, :m2], self.w_imag[:m1, :m2])
        out_ft = tf.einsum('bhwi,hwio->bhwo', x_ft[:, :m1, :m2, :], Wc)
        out_ft = tf.pad(out_ft, [[0, 0], [0, H - m1], [0, (W // 2 + 1) - m2], [0, 0]])
        out = tf.signal.irfft2d(tf.transpose(out_ft, [0, 3, 1, 2]), fft_length=[H, W])
        return tf.transpose(out, [0, 2, 3, 1])


def fno_block(x, filters, modes=12):
    skip = x
    spec = SpectralConv2D(filters, modes=modes)(x)
    local = Conv2D(filters, 1, padding='same')(x)
    x = Add()([spec, local])
    x = BatchNormalization()(x); x = Activation('relu')(x)
    return Add()([x, skip])


def build_fno(filters=12, n_blocks=8):
    x_in = Input(shape=(64, 64, 2))
    x = Conv2DTranspose(filters, (5, 5), strides=(2, 2), padding='same')(x_in)
    for _ in range(n_blocks):
        x = fno_block(x, filters)
    x = Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same')(x)
    return Model(x_in, x, name='FNO')


def load_baselines():
    t = Resnet(input_shape=(64, 64, 2)); t.load_weights(TEACHER_W); t.trainable = False
    s = build_pruned_resnet(8, name='Student'); s.load_weights(STUDENT_W); s.trainable = False
    f = build_fno(); f.load_weights(FNO_W); f.trainable = False
    return {'Teacher': t, 'Student': s, 'FNO': f}


def _wrap(a):
    return (a + np.pi) % (2 * np.pi) - np.pi


def classical_dft_sic(Y, L, NDFT):
    """2D-DFT + successive interference cancellation, same algorithm as Classical_Baseline_SanityTest.ipynb."""
    r = np.fft.ifft2(Y).copy()
    m = np.arange(Q).reshape(-1, 1); n = np.arange(P).reshape(1, -1)
    k1s, k2s = [], []
    for _ in range(L):
        X = np.fft.fft2(r, s=(NDFT, NDFT))
        k = np.unravel_index(np.argmax(np.abs(X)), X.shape)
        k1s.append(k[0]); k2s.append(k[1])
        u = _wrap(-2 * np.pi * k[0] / NDFT); v = _wrap(2 * np.pi * k[1] / NDFT)
        r = r - (X[k] / (Q * P)) * np.exp(-1j * m * u) * np.exp(1j * n * v)
    u = _wrap(-2 * np.pi * np.asarray(k1s) / NDFT); v = _wrap(2 * np.pi * np.asarray(k2s) / NDFT)
    return np.arccos(np.clip(u / np.pi, -1, 1)), np.arccos(np.clip(v / np.pi, -1, 1))


BASE = load_baselines() if not BANKS_ONLY else {}
for k, m in BASE.items():
    log(f'baseline {k}: {m.count_params():,} params')

In [ ]:
# ---------------------------------------------------------------- 9. IABR-Net
class CircularPad(Layer):
    def __init__(self, pad=1, **kw):
        super().__init__(**kw); self.pad = pad

    def call(self, x):
        p = self.pad
        x = tf.concat([x[:, -p:], x, x[:, :p]], axis=1)
        return tf.concat([x[:, :, -p:], x, x[:, :, :p]], axis=2)


def _complex_const(a):
    return tf.constant(a.astype(np.complex64))


class FrontEnd(Layer):
    """mode='iabc': inverse codebook -> IABC-v2 per-element correction -> forward codebook (report + Addendum A1).
    mode='none': identity (Abl-1). mode='dense': Dense(512->512) on the raw input (E2 front-end-swap control)."""
    def __init__(self, mode='iabc', **kw):
        super().__init__(**kw)
        self.mode = mode
        if mode == 'iabc':
            self.ctx = KL.Dense(8, activation='relu', name='iabc_ctx')                  # 4 -> 8
            self.h1 = KL.Dense(16, activation='relu', name='iabc_mlp1')                 # 10 -> 16 (shared over elements)
            self.h2 = KL.Dense(2, kernel_initializer='zeros', name='iabc_mlp2')         # 16 -> 2, starts as identity
        elif mode == 'dense':
            self.dense = KL.Dense(2 * Q * P, name='dense_frontend')                     # 262,656 params
        self.Wi = _complex_const(W_IDEAL); self.Fi = _complex_const(F_IDEAL)
        self.cos_r = tf.constant(COS_R.astype(np.float32)); self.cos_t = tf.constant(COS_T.astype(np.float32))
        self.k = tf.constant(np.arange(16, dtype=np.float32))

    @staticmethod
    def _detrended_phase(z):
        d = tf.math.angle(z[:, 1:] * tf.math.conj(z[:, :-1]))
        ph = tf.concat([tf.zeros_like(d[:, :1]), tf.cumsum(d, axis=1)], axis=1)
        n = tf.range(16, dtype=tf.float32)[None]
        nc = n - tf.reduce_mean(n); pc = ph - tf.reduce_mean(ph, axis=1, keepdims=True)
        slope = tf.reduce_sum(nc * pc, axis=1, keepdims=True) / tf.reduce_sum(nc * nc)
        return pc - slope * nc

    def element_features(self, Ht, Y):
        Er = tf.reduce_sum(tf.abs(Ht) ** 2, axis=2); Et = tf.reduce_sum(tf.abs(Ht) ** 2, axis=1)
        mag_r = 0.5 * tf.math.log(Er / (tf.reduce_mean(Er, 1, keepdims=True) + 1e-12) + 1e-12)
        mag_t = 0.5 * tf.math.log(Et / (tf.reduce_mean(Et, 1, keepdims=True) + 1e-12) + 1e-12)
        idx = tf.argmax(tf.reshape(tf.abs(Y), [-1, Q * P]), axis=1, output_type=tf.int32)
        u_r = tf.gather(self.cos_r, idx // P); u_t = tf.gather(self.cos_t, idx % P)
        ph_r = np.pi * self.k[None] * u_r[:, None]; ph_t = -np.pi * self.k[None] * u_t[:, None]
        rot_r = tf.exp(tf.complex(tf.zeros_like(ph_r), ph_r))
        rot_t = tf.exp(tf.complex(tf.zeros_like(ph_t), ph_t))
        R = Ht * rot_r[:, :, None] * rot_t[:, None, :]
        res_r = self._detrended_phase(tf.reduce_sum(R, axis=2)); res_t = self._detrended_phase(tf.reduce_sum(R, axis=1))
        feat = tf.concat([tf.stack([mag_r, res_r], -1), tf.stack([mag_t, res_t], -1)], axis=1)   # (B,32,2)
        return tf.stop_gradient(feat)

    def call(self, y_ri):
        if self.mode == 'none':
            return y_ri
        if self.mode == 'dense':
            return tf.reshape(self.dense(tf.reshape(y_ri, [-1, 2 * Q * P])), [-1, Q, P, 2])
        Y = tf.complex(y_ri[..., 0], y_ri[..., 1])
        Ht = tf.einsum('nq,bqp,mp->bnm', self.Wi, Y, tf.math.conj(self.Fi))
        feat = self.element_features(Ht, Y)
        mean = tf.reduce_mean(feat, axis=1); std = tf.math.reduce_std(feat, axis=1)
        ctx = self.ctx(tf.concat([mean, std], -1))                                          # (B,8)
        x = tf.concat([feat, tf.tile(ctx[:, None, :], [1, 32, 1])], -1)                   # (B,32,10)
        out = self.h2(self.h1(x))                                                          # (B,32,2): phase, log-gain
        c = tf.exp(tf.complex(-out[..., 1], -out[..., 0]))
        Hc = Ht * c[:, :16, None] * c[:, None, 16:]
        Yc = tf.einsum('nq,bnm,mp->bqp', tf.math.conj(self.Wi), Hc, self.Fi)
        return tf.stack([tf.math.real(Yc), tf.math.imag(Yc)], -1)


class SEResBlock(Layer):
    def __init__(self, c=32, use_se=True, **kw):
        super().__init__(**kw)
        self.use_se = use_se
        self.p1, self.p2 = CircularPad(1), CircularPad(1)
        self.c1, self.c2 = Conv2D(c, 3, padding='valid'), Conv2D(c, 3, padding='valid')
        self.b1, self.b2 = BatchNormalization(), BatchNormalization()
        if use_se:
            self.s1, self.s2 = KL.Dense(c // 8, activation='relu'), KL.Dense(c, activation='sigmoid')

    def call(self, x, training=False):
        h = tf.nn.relu(self.b1(self.c1(self.p1(x)), training=training))
        h = self.b2(self.c2(self.p2(h)), training=training)
        if self.use_se:
            g = self.s2(self.s1(tf.reduce_mean(h, axis=[1, 2])))
            h = h * g[:, None, None, :]
            gm = tf.reduce_mean(g, axis=1)
        else:
            gm = tf.zeros_like(h[:, 0, 0, 0])
        return tf.nn.relu(h + x), gm


class IABRNet(Model):
    def __init__(self, front='iabc', use_se=True, n_blocks=10, c=32, k_slots=10, **kw):
        super().__init__(**kw)
        self.front_mode, self.use_se, self.K = front, use_se, k_slots
        self.front = FrontEnd(front)
        self.stem_pad, self.stem = CircularPad(1), Conv2D(c, 3, padding='valid', activation='relu')
        self.blocks = [SEResBlock(c, use_se) for _ in range(n_blocks)]
        self.token = Conv2D(c, 1, activation='relu')
        self.heat = Conv2D(1, 1, bias_initializer=tf.keras.initializers.Constant(-2.19))
        self.crop_pad = CircularPad(3)
        self.aoa1, self.aoa2 = KL.Dense(16, activation='relu'), KL.Dense(1)
        self.aod1, self.aod2 = KL.Dense(16, activation='relu'), KL.Dense(1)
        self.snr_head = KL.Dense(1) if use_se else None
        g = np.stack(np.meshgrid(np.arange(7), np.arange(7), indexing='ij'), -1).astype(np.int32)
        self.win = tf.constant(g)                                                           # (7,7,2)

    def backbone(self, y_ri, training=False):
        yc = self.front(y_ri)
        x = self.stem(self.stem_pad(yc))
        gms = []
        for b in self.blocks:
            x, gm = b(x, training=training); gms.append(gm)
        tok = self.token(x)
        return tok, self.heat(tok)[..., 0], tf.stack(gms, 1), yc

    def refine(self, tok, idx):
        B = tf.shape(tok)[0]
        tp = self.crop_pad(tok)                                                             # (B,22,22,C)
        q0 = idx // P; p0 = idx % P
        rows = q0[:, :, None, None] + self.win[None, None, :, :, 0]
        cols = p0[:, :, None, None] + self.win[None, None, :, :, 1]
        bb = tf.broadcast_to(tf.range(B)[:, None, None, None], tf.shape(rows))
        crops = tf.gather_nd(tp, tf.stack([bb, rows, cols], -1))                            # (B,K,7,7,C)
        K = tf.shape(idx)[1]
        a = tf.reshape(self.aoa1(crops), [B, K, 7 * 7 * 16]); d = tf.reshape(self.aod1(crops), [B, K, 7 * 7 * 16])
        return self.aoa2(a)[..., 0], self.aod2(d)[..., 0]

    def call(self, inputs, training=False):
        y_ri, idx = inputs
        tok, hl, gms, yc = self.backbone(y_ri, training)
        dq, dp = self.refine(tok, idx)
        snr = self.snr_head(gms)[:, 0] if self.use_se else tf.zeros_like(dq[:, 0])
        return dict(heat_logits=hl, dq=dq, dp=dp, snr=snr, y_corr=yc)


def build_iabr(front='iabc', use_se=True):
    m = IABRNet(front=front, use_se=use_se, k_slots=CFG['k_slots'])
    m((tf.zeros((1, Q, P, 2)), tf.zeros((1, CFG['k_slots']), tf.int32)))
    return m


VARIANTS = {
    # name: (front, use_se, lambdas override, steps)
    'IABR_s0': ('iabc', True, {}, 'full'),
    'IABR_s1': ('iabc', True, {}, 'full'),
    'IABR_s2': ('iabc', True, {}, 'full'),
    'E1_capacity_control': ('iabc', True, dict(pair=0.0), 'full'),
    'E2_dense_frontend': ('dense', True, dict(pair=0.0), 'full'),
    'Abl1_no_IABC': ('none', True, dict(pair=0.0), 'full'),
    'Abl3_no_SE': ('iabc', False, dict(snr=0.0), 'full'),
}
VARIANTS = {k: v for k, v in VARIANTS.items() if not k.startswith('IABR_s') or int(k[-1]) in CFG['seeds']}
for l3 in CFG['sweep_l3']:
    for l4 in CFG['sweep_l4']:
        VARIANTS[f'Abl6_l3_{l3}_l4_{l4}'] = ('iabc', True, dict(pair=l3, snr=l4), 'sweep')
log(f'{len(VARIANTS)} IABR-Net training runs configured:', ', '.join(VARIANTS))

In [ ]:
# ---------------------------------------------------------------- 10. E0b -- real parameter count, FLOPs, memory
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2


def profiler_flops(fn, specs):
    cf = tf.function(fn).get_concrete_function(*specs)
    frozen = convert_variables_to_constants_v2(cf)
    opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
    opts['output'] = 'none'
    prof = tf.compat.v1.profiler.profile(graph=frozen.graph, run_meta=tf.compat.v1.RunMetadata(), cmd='op', options=opts)
    op_types = sorted({op.type for op in frozen.graph.get_operations()})
    counted = {c.name: int(c.float_ops) for c in prof.children}
    return int(prof.total_float_ops), counted, op_types


def iabr_infer_fn(model):
    def f(y):
        tok, hl, gms, yc = model.backbone(y, training=False)
        _, idx = tf.math.top_k(tf.reshape(hl, [-1, Q * P]), k=model.K)
        dq, dp = model.refine(tok, idx)
        return hl, dq, dp
    return f


def complex_einsum_flops_iabr():
    # inverse + forward codebook: 4 complex 16x16x16 matmuls; complex MAC = 8 real FLOPs (profiler skips complex einsum)
    return 4 * 16 ** 3 * 8


def fno_fft_flops():
    # the profiler skips FFT and complex einsum: 8 blocks x 12 channels x (rfft2d + irfft2d) on 128x128 at
    # ~2.5 N log2 N real FLOPs each (half spectrum), plus the 12x12-mode complex channel mix (8 real FLOPs per MAC)
    N = 128 * 128
    return int(8 * 12 * 2 * 2.5 * N * np.log2(N) + 8 * 12 * 12 * 12 * 12 * 8)


def e0b():
    m = build_iabr()
    rows = []
    for layer in m.layers:
        rows.append(dict(layer=layer.name, params=int(layer.count_params())))
    trainable = int(np.sum([np.prod(v.shape) for v in m.trainable_variables]))
    total = int(m.count_params())
    fl, counted, ops = profiler_flops(iabr_infer_fn(m), [tf.TensorSpec([1, Q, P, 2], tf.float32)])
    res = dict(params_total=total, params_trainable=trainable, report_estimate_params=193104,
               flops_profiler=fl, flops_complex_supplement=complex_einsum_flops_iabr(),
               flops_total=fl + complex_einsum_flops_iabr(), report_estimate_flops=95.3e6,
               profiler_counted_ops=counted, graph_op_types=ops, per_layer=rows,
               param_memory_MB=total * 4 / 2 ** 20)
    log(f'  IABR-Net params: {total:,} (trainable {trainable:,}) vs report estimate 193,104')
    log(f'  IABR-Net FLOPs (batch 1): {res["flops_total"] / 1e6:.2f} M vs report estimate 95.3 M')
    return res


R_E0B = run_exp('E0b_model_instantiation', e0b)

## Training

Every run trains on fresh synthetic data generated on the fly. Each sample has L ∈ {1..9} paths,
SNR ∈ {−15..24} dB, and per-sample random impairment (phase δmax ∈ U(0, 5°), gain γmax ∈ U(0, 2) dB).

- Each training sample also comes with its **paired clean observation** (the same channel and the
  same noise, but ideal codebooks). This is the target of `L_pair`.
- **Resume:** each run checkpoints the model and optimizer every `ckpt_every` steps, and writes a
  `done.json` when it finishes. Re-running the notebook continues from the last checkpoint.
- After the first few hundred steps it logs a **measured ETA** for the run.

**Controlled comparisons.** All IABR-Net variants use the *same* impairment-augmented data, so
IABR vs E1, Abl-1, E2 and Abl-3 isolates the architecture. Teacher, Student and FNO were trained on
clean data by their original pipelines. Their robustness numbers are reference points, not
controlled comparisons.

In [ ]:
# ---------------------------------------------------------------- 11. Data pipeline + losses + training loop
def coarse_targets(psi, phi, L_arr, K, sigma_c):
    B = len(L_arr)
    heat = np.zeros((B, Q, P), np.float32)
    idx = np.zeros((B, K), np.int32); off = np.zeros((B, K, 2), np.float32); mask = np.zeros((B, K), np.float32)
    rr = np.arange(Q)[:, None]; cc = np.arange(P)[None, :]
    for b in range(B):
        for l in range(int(L_arr[b])):
            qs, ps = angles_to_cells(psi[b, l], phi[b, l])
            qi, pi_ = int(np.round(qs)) % Q, int(np.round(ps)) % P
            dq = np.minimum(np.abs(rr - qi), Q - np.abs(rr - qi)); dp = np.minimum(np.abs(cc - pi_), P - np.abs(cc - pi_))
            heat[b] = np.maximum(heat[b], np.exp(-(dq ** 2 + dp ** 2) / (2 * sigma_c ** 2)))
            if l < K:
                idx[b, l] = qi * P + pi_
                off[b, l] = (qs - np.round(qs), ps - np.round(ps))
                mask[b, l] = 1.0
    return heat, idx, off, mask


def make_batch(rng, B):
    L = rng.integers(CFG['train_L'][0], CFG['train_L'][1] + 1, B)
    snr = rng.integers(CFG['train_snr'][0], CFG['train_snr'][1] + 1, B).astype(float)
    pd = rng.uniform(0, CFG['train_phase_deg_max'], B); gd = rng.uniform(0, CFG['train_gain_db_max'], B)
    psi, phi, alpha = sample_paths(L, rng, lmax=CFG['train_L'][1])
    sim = simulate(psi, phi, alpha, snr, rng, phase_deg_max=pd, gain_db_max=gd, want_clean=True)
    heat, idx, off, mask = coarse_targets(psi, phi, L, CFG['k_slots'], CFG['heat_sigma_cells'])
    return (to_ri(sim['Y']), to_ri(sim['Y_clean']), heat, idx, off, mask, (snr / 10).astype(np.float32))


SIG = (tf.TensorSpec([None, Q, P, 2], tf.float32), tf.TensorSpec([None, Q, P, 2], tf.float32),
       tf.TensorSpec([None, Q, P], tf.float32), tf.TensorSpec([None, None], tf.int32),
       tf.TensorSpec([None, None, 2], tf.float32), tf.TensorSpec([None, None], tf.float32), tf.TensorSpec([None], tf.float32))


def make_dataset(seed):
    def gen(worker):
        rng = np.random.default_rng([seed, int(worker)])
        while True:
            yield make_batch(rng, CFG['batch'])
    nw = CFG['data_workers']
    ds = tf.data.Dataset.range(nw).interleave(
        lambda w: tf.data.Dataset.from_generator(gen, args=(w,), output_signature=SIG),
        cycle_length=nw, num_parallel_calls=nw, deterministic=True)
    return ds.prefetch(4)


def focal_loss(logits, target):
    p = tf.clip_by_value(tf.sigmoid(logits), 1e-4, 1 - 1e-4)
    pos = tf.cast(target >= 1.0 - 1e-6, tf.float32)
    pos_l = tf.math.log(p) * (1 - p) ** 2 * pos
    neg_l = tf.math.log(1 - p) * p ** 2 * (1 - target) ** 4 * (1 - pos)
    return -(tf.reduce_sum(pos_l) + tf.reduce_sum(neg_l)) / tf.maximum(tf.reduce_sum(pos), 1.0)


def lr_at(step, total):
    if step < CFG['warmup']:
        return CFG['lr_max'] * (step + 1) / CFG['warmup']
    t = (step - CFG['warmup']) / max(1, total - CFG['warmup'])
    return CFG['lr_min'] + 0.5 * (CFG['lr_max'] - CFG['lr_min']) * (1 + math.cos(math.pi * min(1.0, t)))


def train_variant(name):
    front, use_se, lam_over, kind = VARIANTS[name]
    lam = dict(CFG['lambdas']); lam.update(lam_over)
    if front != 'iabc':
        lam['pair'] = 0.0
    if not use_se:
        lam['snr'] = 0.0
    steps = CFG['full_steps'] if kind == 'full' else CFG['sweep_steps']
    seed = int(name[-1]) if name.startswith('IABR_s') else 0
    cdir = os.path.join(DIRS['checkpoints'], name)
    os.makedirs(cdir, exist_ok=True)
    wpath = os.path.join(cdir, 'final.weights.h5'); dpath = os.path.join(cdir, 'done.json')
    if os.path.exists(dpath) and os.path.exists(wpath):
        return json.load(open(dpath))
    tf.keras.utils.set_random_seed(1000 + seed)
    model = build_iabr(front, use_se)
    opt = tf.keras.optimizers.Adam(CFG['lr_max'], global_clipnorm=CFG['clipnorm'])
    step_var = tf.Variable(0, dtype=tf.int64)
    ckpt = tf.train.Checkpoint(model=model, optimizer=opt, step=step_var)
    mgr = tf.train.CheckpointManager(ckpt, os.path.join(cdir, 'ckpt'), max_to_keep=2)
    hist_path = os.path.join(cdir, 'history.json')
    history = json.load(open(hist_path)) if os.path.exists(hist_path) else []
    if mgr.latest_checkpoint:
        ckpt.restore(mgr.latest_checkpoint)
        log(f'  [{name}] resumed at step {int(step_var.numpy())}')

    @tf.function
    def train_step(y, yc, heat, idx, off, mask, snr):
        with tf.GradientTape() as tape:
            o = model((y, idx), training=True)
            l_heat = focal_loss(o['heat_logits'], heat)
            err = tf.stack([o['dq'], o['dp']], -1) - off
            sl1 = tf.where(tf.abs(err) < 1.0, 0.5 * tf.square(err), tf.abs(err) - 0.5)          # smooth-L1
            l_off = tf.reduce_sum(tf.reduce_sum(sl1, -1) * mask) / tf.maximum(tf.reduce_sum(mask), 1.0)
            num = tf.reduce_sum(tf.square(o['y_corr'] - yc), axis=[1, 2, 3])
            den = tf.reduce_sum(tf.square(yc), axis=[1, 2, 3]) + 1e-6
            l_pair = tf.reduce_mean(num / den)
            l_snr = tf.reduce_mean(tf.square(o['snr'] - snr))
            loss = lam['heat'] * l_heat + lam['off'] * l_off + lam['pair'] * l_pair + lam['snr'] * l_snr
        g = tape.gradient(loss, model.trainable_variables)
        opt.apply_gradients(zip(g, model.trainable_variables))
        return loss, l_heat, l_off, l_pair, l_snr

    it = iter(make_dataset(seed))
    start = int(step_var.numpy())
    t0 = time.time(); acc = np.zeros(5); n_acc = 0; eta_logged = False; t_first_log = None
    for step in range(start, steps):
        opt.learning_rate.assign(lr_at(step, steps))
        out = train_step(*next(it))
        acc += np.array([float(v) for v in out]); n_acc += 1
        step_var.assign(step + 1)
        if (step + 1) % CFG['log_every'] == 0 or step + 1 == steps:
            a = acc / n_acc
            history.append(dict(step=step + 1, loss=a[0], heat=a[1], off=a[2], pair=a[3], snr=a[4], lr=lr_at(step, steps)))
            acc[:] = 0; n_acc = 0
            if t_first_log is None:
                t_first_log = time.time()          # first window includes graph tracing -> measure from the second one
            elif not eta_logged:
                sps = (time.time() - t_first_log) / CFG['log_every']
                log(f'  [{name}] measured {sps * 1000:.1f} ms/step -> ETA {sps * (steps - step - 1) / 60:.1f} min for this run')
                eta_logged = True
            if (step + 1) % (CFG['log_every'] * 5) == 0 or step + 1 == steps:
                log(f'  [{name}] step {step + 1}/{steps} loss={a[0]:.4f} (heat {a[1]:.4f} off {a[2]:.4f} pair {a[3]:.4f} snr {a[4]:.4f})')
        if (step + 1) % CFG['ckpt_every'] == 0:
            mgr.save(); save_json(history, hist_path)
    model.save_weights(wpath)
    save_json(history, hist_path)
    info = dict(name=name, front=front, use_se=use_se, lambdas=lam, steps=steps, seed=seed,
                params=int(model.count_params()), train_min=(time.time() - t0) / 60, final=history[-1] if history else None)
    save_json(info, dpath)
    return info


def load_variant(name):
    front, use_se, _, _ = VARIANTS[name]
    m = build_iabr(front, use_se)
    m.load_weights(os.path.join(DIRS['checkpoints'], name, 'final.weights.h5'))
    return m

In [ ]:
# ---------------------------------------------------------------- 12. Train every variant
TRAIN = {}
for name in VARIANTS:
    r = run_exp(f'TRAIN_{name}', lambda n=name: train_variant(n), needs=['E0a_phase0_unit_tests'])
    TRAIN[name] = r


def trained(name):
    return str(STATUS.get(f'TRAIN_{name}', '')).startswith('done')


if TRAIN and not BANKS_ONLY:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for name in VARIANTS:
        hp = os.path.join(DIRS['checkpoints'], name, 'history.json')
        if not os.path.exists(hp):
            continue
        h = json.load(open(hp))
        if not h:
            continue
        st = [x['step'] for x in h]
        tgt = ax[1] if name.startswith('Abl6') else ax[0]
        tgt.plot(st, [x['loss'] for x in h], label=name)
    for a, t in zip(ax, ['Main runs + controls', 'Abl-6 loss-weight sweep']):
        a.set_xlabel('step'); a.set_ylabel('total loss'); a.set_title(t); a.grid(alpha=.3); a.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(os.path.join(DIRS['figures'], 'training_loss.png'), dpi=130); plt.show()

## Evaluation engine

Every model is scored with the project's **original** matching and metric code
(`prepare_for_metric` → `get_ang_difference` → `filter_angles`, 1° threshold). The following are
reported for every model and condition:

| Metric | Definition |
|---|---|
| `pd_paper` | The original convention. Component-level; samples with fewer than L detections are **dropped**. Directly comparable to every past number in this project. |
| `pd_strict` | Same, but dropped samples count as misses. The report requires this on every table. |
| `pd_source` | A path counts only if **both** its AoA and AoD are within 1°. Missed paths count. |
| `pd_aoa` / `pd_aod`, `rmse_aoa` / `rmse_aod` / `rmse` | Split by component. RMSE is over detected components (paper convention). |
| `p50` / `p95` | Absolute error over all matched components. |
| `p95_endfire` / `p95_broadside` | End-fire = true angle within 30° of 0° or 180°. This is the Gap-5 probe. |
| `pairing_error` | Share of matched paths where exactly one of the two components is within 1°. |

Predictions are cached in `outputs/cache/`, so re-running the tables costs nothing.

In [ ]:
# ---------------------------------------------------------------- 13. Prediction functions + metrics
DETECTOR = get_blob_detector()
_FWD = {}


def _fwd(model):
    if id(model) not in _FWD:
        _FWD[id(model)] = tf.function(lambda x: model(x, training=False), reduce_retracing=True)
    return _FWD[id(model)]


def predict_heatmap(model, bank):
    Y16, Ls = bank['Y16'], bank['L']
    out = []
    bs = CFG['eval_batch']
    for s in range(0, len(Y16), bs):
        pred = _fwd(model)(tf.constant(upsample64(Y16[s:s + bs])))
        for j in range(pred.shape[0]):
            peaks, amps = get_blob_peaks(pred[j], DETECTOR)
            peaks = peaks[np.argsort(-amps)[:int(Ls[s + j])]]
            out.append(peaks_to_angles(peaks, sigma=SIGMA_GT, grid_size=M_GT))
    return out


def predict_dft_sic(bank):
    Y = bank['Y16'][..., 0] + 1j * bank['Y16'][..., 1]
    return [classical_dft_sic(Y[i], int(bank['L'][i]), CFG['dft_ndft']) for i in range(len(Y))]


_IABR_FN = {}


def _iabr_fns(model):
    if id(model) not in _IABR_FN:
        bb = tf.function(lambda y: model.backbone(y, training=False)[:2], reduce_retracing=True)
        rf = tf.function(lambda t, i: model.refine(t, i), reduce_retracing=True)
        _IABR_FN[id(model)] = (bb, rf)
    return _IABR_FN[id(model)]


def local_maxima(heat, kmax):
    mx = scipy.ndimage.maximum_filter(heat, size=3, mode='wrap')
    cand = np.argwhere(heat >= mx)
    vals = heat[cand[:, 0], cand[:, 1]]
    order = np.argsort(-vals)[:kmax]
    return cand[order]


def parabolic(h, a, b, axis):
    if axis == 0:
        m1, c0, p1 = h[(a - 1) % Q, b], h[a, b], h[(a + 1) % Q, b]
    else:
        m1, c0, p1 = h[a, (b - 1) % P], h[a, b], h[a, (b + 1) % P]
    den = m1 - 2 * c0 + p1
    return float(np.clip(0.5 * (m1 - p1) / den, -0.5, 0.5)) if abs(den) > 1e-12 else 0.0


def predict_iabr(model, bank, decode='learned'):
    bb, rf = _iabr_fns(model)
    K = CFG['k_slots']
    Y16, Ls = bank['Y16'], bank['L']
    out = []
    bs = max(CFG['eval_batch'], 256) if not SMOKE_TEST else CFG['eval_batch']
    for s in range(0, len(Y16), bs):
        tok, hl = bb(tf.constant(Y16[s:s + bs]))
        heat = tf.sigmoid(hl).numpy()
        cells = [local_maxima(heat[j], K) for j in range(heat.shape[0])]
        idx = np.zeros((heat.shape[0], K), np.int32)
        for j, c in enumerate(cells):
            flat = c[:, 0] * P + c[:, 1]
            idx[j, :len(flat)] = flat
            idx[j, len(flat):] = flat[0] if len(flat) else 0
        dq, dp = [v.numpy() for v in rf(tok, tf.constant(idx))]
        for j, c in enumerate(cells):
            L = int(Ls[s + j]); c = c[:L]
            if decode == 'learned':
                qs = c[:, 0] + dq[j, :len(c)]; ps = c[:, 1] + dp[j, :len(c)]
            elif decode == 'parabolic':
                qs = np.array([a + parabolic(heat[j], a, b, 0) for a, b in c], float)
                ps = np.array([b + parabolic(heat[j], a, b, 1) for a, b in c], float)
            else:
                qs, ps = c[:, 0].astype(float), c[:, 1].astype(float)
            out.append(cells_to_angles(qs, ps) if len(c) else (np.array([]), np.array([])))
    return out


def predict(model_name, bank_name, decode='learned'):
    tag = model_name if decode == 'learned' else f'{model_name}__{decode}'
    cpath = os.path.join(DIRS['cache'], f'pred__{tag}__{bank_name}.npz')
    bank = load_bank(bank_name)
    lmax = int(np.max(bank['L']))
    if os.path.exists(cpath):
        z = np.load(cpath)
        return [(z['psi'][i, :z['n'][i]], z['phi'][i, :z['n'][i]]) for i in range(len(z['n']))]
    if model_name in BASE:
        preds = predict_heatmap(BASE[model_name], bank)
    elif model_name == 'DFT-SIC':
        preds = predict_dft_sic(bank)
    else:
        preds = predict_iabr(get_iabr(model_name), bank, decode)
    n = np.array([len(p[0]) for p in preds]); ps = np.full((len(preds), lmax), np.nan); ph = np.full((len(preds), lmax), np.nan)
    for i, p in enumerate(preds):
        ps[i, :n[i]] = p[0][:lmax]; ph[i, :n[i]] = p[1][:lmax]
    np.savez_compressed(cpath, psi=ps, phi=ph, n=n)
    return preds


_IABR_MODELS = {}


def get_iabr(name):
    if name not in _IABR_MODELS:
        _IABR_MODELS[name] = load_variant(name)
    return _IABR_MODELS[name]


def metrics(preds, bank, idx=None, sub=None):
    """sub: optional slice of path indices to score (e.g. principal paths 0..2 of the nuisance banks)."""
    idx = np.arange(len(preds)) if idx is None else idx
    good_c = bad_c = 0; strict_bad = 0; dropped = 0
    comp_good = {0: 0, 1: 0}; comp_tot = {0: 0, 1: 0}; comp_err = {0: [], 1: []}
    src_ok = src_tot = pair_err = matched_paths = 0
    all_abs, end_abs, broad_abs, good_vals = [], [], [], []
    for i in idx:
        L = int(bank['L'][i])
        feat = np.stack([bank['psi'][i, :L], bank['phi'][i, :L]]).astype(np.float64)
        gt, pr = prepare_for_metric(preds[i], feat)
        sel = np.arange(L) if sub is None else np.arange(L)[sub]
        if np.isnan(pr).any():
            dropped += 1; strict_bad += 2 * len(sel); src_tot += len(sel)
            continue
        dif = get_ang_difference(gt, pr, return_flat=False)[:, sel]
        ok = np.abs(dif) <= 1.0
        good_c += int(ok.sum()); bad_c += int((~ok).sum())
        good_vals += list(dif[ok])
        for c in (0, 1):
            comp_good[c] += int(ok[c].sum()); comp_tot[c] += ok.shape[1]; comp_err[c] += list(dif[c][ok[c]])
        both = ok.all(0); src_ok += int(both.sum()); src_tot += len(sel)
        pair_err += int((ok.sum(0) == 1).sum()); matched_paths += len(sel)
        for c in (0, 1):
            for l, e in zip(sel, dif[c]):
                a = abs(e); all_abs.append(a)
                (end_abs if abs(np.cos(gt[c, l])) > np.cos(np.deg2rad(30)) else broad_abs).append(a)
    tot = good_c + bad_c
    q = lambda v, p: float(np.percentile(v, p)) if len(v) else np.nan
    rm = lambda v: float(np.sqrt(np.mean(np.square(v)))) if len(v) else np.nan
    return dict(pd_paper=good_c / tot if tot else np.nan,
                pd_strict=good_c / (tot + strict_bad) if (tot + strict_bad) else np.nan,
                pd_source=src_ok / src_tot if src_tot else np.nan,
                pd_aoa=comp_good[0] / comp_tot[0] if comp_tot[0] else np.nan,
                pd_aod=comp_good[1] / comp_tot[1] if comp_tot[1] else np.nan,
                rmse=rm(good_vals), rmse_aoa=rm(comp_err[0]), rmse_aod=rm(comp_err[1]),
                p50=q(all_abs, 50), p95=q(all_abs, 95), p95_endfire=q(end_abs, 95), p95_broadside=q(broad_abs, 95),
                pairing_error=pair_err / matched_paths if matched_paths else np.nan,
                n_samples=len(idx), n_dropped=dropped)


def by_snr(preds, bank):
    out = {}
    for s in sorted(set(bank['snr'].astype(int).tolist())):
        out[s] = metrics(preds, bank, np.where(bank['snr'].astype(int) == s)[0])
    keys = ['pd_paper', 'pd_strict', 'pd_source', 'rmse', 'rmse_aoa', 'rmse_aod', 'p95', 'p95_endfire']
    out['mean_over_snr'] = {k: float(np.nanmean([out[s][k] for s in out if s != 'mean_over_snr'])) for k in keys}
    return out


def available_iabr(names):
    return [n for n in names if n in VARIANTS and trained(n)]


def md_table(rows, cols, fmt=None):
    fmt = fmt or {}
    head = '| ' + ' | '.join(cols) + ' |\n|' + '---|' * len(cols) + '\n'
    body = ''
    for r in rows:
        cells = []
        for c in cols:
            v = r.get(c, '')
            if isinstance(v, (float, np.floating)):
                v = ('—' if not np.isfinite(v) else (fmt.get(c, '{:.4f}').format(v)))
            elif isinstance(v, (int, np.integer)) and not isinstance(v, bool) and c in fmt:
                v = fmt[c].format(v)
            cells.append(str(v))
        body += '| ' + ' | '.join(cells) + ' |\n'
    return head + body


TABLES = {}


def put_table(key, title, text):
    TABLES[key] = (title, text)
    with open(os.path.join(DIRS['tables'], f'{key}.md'), 'w', encoding='utf-8') as f:
        f.write(f'### {title}\n\n{text}\n')

In [ ]:
# ---------------------------------------------------------------- 14. V0 + V1 -- reproducibility checks
def v0():
    preds = predict('Teacher', 'E3_frozen')
    r = by_snr(preds, load_bank('E3_frozen'))
    mean_pd = r['mean_over_snr']['pd_paper']
    ref = 0.7103495885388549
    # exact on the CPU/GPU runs so far; a different batch size / cuDNN kernel may flip a handful of the 48,000
    # components, so allow 2e-3 -- anything larger means a real mismatch (weights, data, or evaluator)
    ok = SMOKE_TEST or abs(mean_pd - ref) < 2e-3
    log(f'  V0 teacher mean Pd on frozen bank = {mean_pd:.10f} (registry {ref}, diff {mean_pd - ref:+.2e}) -> {"PASS" if ok else "FAIL"}')
    assert ok, 'Teacher does not reproduce the registry number on the frozen bank'
    return dict(teacher_mean_pd=mean_pd, registry=ref, diff=mean_pd - ref, passed=ok, per_snr=r)


def v1():
    fb = by_snr(predict('Teacher', 'E3_frozen'), load_bank('E3_frozen'))
    rows = []
    for s in [0, 15, 25]:
        b = load_bank(f'v1_clean_snr{s}')
        mine = metrics(predict('Teacher', f'v1_clean_snr{s}'), b)['pd_paper']
        ref = fb[s]['pd_paper']; se = math.sqrt(max(ref * (1 - ref), 1e-9) / (6 * len(b['L'])))
        rows.append(dict(snr=s, frozen_bank_pd=ref, new_generator_pd=mine, diff=mine - ref, three_sigma=3 * se,
                         ok=bool(abs(mine - ref) <= max(3 * se, 0.02) or SMOKE_TEST)))
        log(f'  V1 SNR {s:>2} dB: frozen {ref:.4f} vs new generator {mine:.4f} (|diff| {abs(mine - ref):.4f}, 3sigma {3 * se:.4f})')
    put_table('V1', 'V1 — new-bank generator fidelity (Teacher Pd, frozen bank vs regenerated clean L=3 bank)',
              md_table(rows, ['snr', 'frozen_bank_pd', 'new_generator_pd', 'diff', 'three_sigma', 'ok']))
    return dict(rows=rows, all_ok=all(r['ok'] for r in rows))


R_V0 = run_exp('V0_teacher_reproduces_registry', v0)
R_V1 = run_exp('V1_generator_fidelity', v1, needs=['BANKS_build'])

In [ ]:
# ---------------------------------------------------------------- 15. E3 -- in-distribution SNR sweep (full frozen bank)
MAIN = 'IABR_s0'


def e3():
    bank = load_bank('E3_frozen')
    res = {}
    for m in ['Teacher', 'Student', 'FNO', 'DFT-SIC'] + available_iabr([k for k in VARIANTS if not k.startswith('Abl6')]):
        res[m] = by_snr(predict(m, 'E3_frozen'), bank)
        log(f'  E3 {m:<24} mean Pd(paper) {res[m]["mean_over_snr"]["pd_paper"]:.4f} strict {res[m]["mean_over_snr"]["pd_strict"]:.4f}')
    snrs = [s for s in res['Teacher'] if s != 'mean_over_snr']
    rows = []
    for m, r in res.items():
        row = dict(model=m)
        for s in snrs:
            row[f'{s}dB'] = r[s]['pd_paper']
        row['mean Pd'] = r['mean_over_snr']['pd_paper']; row['mean strict'] = r['mean_over_snr']['pd_strict']
        row['mean source'] = r['mean_over_snr']['pd_source']; row['RMSE AoA'] = r['mean_over_snr']['rmse_aoa']
        row['RMSE AoD'] = r['mean_over_snr']['rmse_aod']
        rows.append(row)
    put_table('E3', f'E3 — in-distribution SNR sweep, paper-style Pd per SNR + mean metrics ({len(bank["L"])} frozen-bank samples)',
              md_table(rows, ['model'] + [f'{s}dB' for s in snrs] + ['mean Pd', 'mean strict', 'mean source', 'RMSE AoA', 'RMSE AoD']))
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    for m, r in res.items():
        if m.startswith('Abl6'):
            continue
        for a, k in zip(ax, ['pd_paper', 'pd_strict', 'rmse']):
            a.plot(snrs, [r[s][k] for s in snrs], marker='o', ms=3, label=m)
    for a, t in zip(ax, ['Pd (paper-style)', 'Pd (strict)', 'RMSE of detected (deg)']):
        a.set_xlabel('SNR (dB)'); a.set_title(t); a.grid(alpha=.3)
    ax[0].legend(fontsize=7); plt.tight_layout(); plt.savefig(os.path.join(DIRS['figures'], 'E3_snr_sweep.png'), dpi=130); plt.show()
    return dict(results=res)


R_E3 = run_exp('E3_in_distribution_snr_sweep', e3, needs=['E0a_phase0_unit_tests'])

In [ ]:
# ---------------------------------------------------------------- 16. Abl-2 -- does crop refinement escape the grid ceiling? (end-fire P95)
def abl2():
    bank = load_bank('E3_frozen')
    rows, res = [], {}
    entries = [('Teacher', 'learned'), ('Student', 'learned'), ('FNO', 'learned'), ('DFT-SIC', 'learned')]
    if trained(MAIN):
        entries += [(MAIN, 'coarse'), (MAIN, 'parabolic'), (MAIN, 'learned')]
    for m, dec in entries:
        r = metrics(predict(m, 'E3_frozen', dec), bank)
        label = m if m != MAIN else f'{m} [{"learned refine" if dec == "learned" else dec}]'
        res[label] = r
        rows.append(dict(model=label, **{k: r[k] for k in ['pd_paper', 'pd_strict', 'rmse', 'p50', 'p95', 'p95_broadside', 'p95_endfire']}))
    put_table('Abl2', f'Abl-2 — decode ablation and end-fire tail ({len(bank["L"])} frozen-bank samples pooled; errors in degrees)',
              md_table(rows, ['model', 'pd_paper', 'pd_strict', 'rmse', 'p50', 'p95', 'p95_broadside', 'p95_endfire']))
    return dict(results=res)


R_ABL2 = run_exp('Abl2_refinement_endfire', abl2, needs=['E0a_phase0_unit_tests'])

In [ ]:
# ---------------------------------------------------------------- 17. Robustness sweeps E4 / E5 / E6 / E7 / E8 / SNR tails
ROBUST_MODELS = ['Teacher', 'Student', 'FNO', 'DFT-SIC', MAIN, 'E1_capacity_control', 'Abl1_no_IABC']


def sweep(exp_key, title, banks, xlabel, xvals):
    models = [m for m in ROBUST_MODELS if m in BASE or m == 'DFT-SIC' or trained(m)]
    res = {m: {} for m in models}
    for bname, x in zip(banks, xvals):
        b = load_bank(bname)
        for m in models:
            res[m][bname] = metrics(predict(m, bname), b)
    rows = []
    for m in models:
        row = dict(model=m)
        for bname, x in zip(banks, xvals):
            row[f'{x}'] = res[m][bname]['pd_paper']
        rows.append(row)
    rows_strict = [dict(model=m, **{f'{x}': res[m][b]['pd_strict'] for b, x in zip(banks, xvals)}) for m in models]
    put_table(exp_key, f'{title} — paper-style Pd ({xlabel})', md_table(rows, ['model'] + [f'{x}' for x in xvals]))
    put_table(exp_key + '_strict', f'{title} — strict Pd ({xlabel})', md_table(rows_strict, ['model'] + [f'{x}' for x in xvals]))
    return res, models


def plot_sweep(res, models, banks, xvals, xlabel, fname, key='pd_paper', extra=None):
    plt.figure(figsize=(7, 4.2))
    for m in models:
        plt.plot(range(len(xvals)), [res[m][b][key] for b in banks], marker='o', ms=3, label=m)
    plt.xticks(range(len(xvals)), [str(x) for x in xvals]); plt.xlabel(xlabel); plt.ylabel(key)
    plt.grid(alpha=.3); plt.legend(fontsize=7); plt.title(fname.replace('_', ' ').replace('.png', ''))
    plt.tight_layout(); plt.savefig(os.path.join(DIRS['figures'], fname), dpi=130); plt.show()


def e4():
    out = {}
    for snr in CFG['robust_snrs']:
        banks = [f'phase_d{d}_snr{snr}' for d in [0, 1, 2, 5]]
        r, ms = sweep(f'E4_snr{snr}', f'E4 phase error, SNR {snr} dB', banks, 'phase error max (deg)', [0, 1, 2, 5])
        plot_sweep(r, ms, banks, [0, 1, 2, 5], 'phase error max (deg)', f'E4_phase_snr{snr}.png'); out[snr] = r
    return dict(results=out)


def e5():
    out = {}
    for snr in CFG['robust_snrs']:
        banks = [f'phase_d0_snr{snr}'] + [f'gain_g{g}_snr{snr}' for g in [0.5, 1, 2, 4]]
        xs = [0, 0.5, 1, 2, '4 (OOD)']
        r, ms = sweep(f'E5_snr{snr}', f'E5 gain error, SNR {snr} dB', banks, 'gain error max (dB)', xs)
        plot_sweep(r, ms, banks, xs, 'gain error max (dB)', f'E5_gain_snr{snr}.png'); out[snr] = r
    return dict(results=out)


def e6():
    out = {}
    for snr in CFG['robust_snrs']:
        banks = [f'phase_d5_snr{snr}', f'oodphase_d10_snr{snr}', f'oodphase_d15_snr{snr}']
        r, ms = sweep(f'E6_snr{snr}', f'E6 OOD phase error, SNR {snr} dB', banks, 'phase error max (deg)', ['5 (in)', '10 (OOD)', '15 (OOD)'])
        out[snr] = r
    return dict(results=out)


def e7():
    out = {}
    Ls = [1, 2, 3, 4, 5, 6, 7, 8, 10]
    for snr in CFG['robust_snrs']:
        banks = [f'phase_d0_snr{snr}' if L == 3 else f'L{L}_snr{snr}' for L in Ls]
        xs = [str(L) if L < 10 else '10 (OOD)' for L in Ls]
        r, ms = sweep(f'E7_snr{snr}', f'E7 path count, SNR {snr} dB', banks, 'number of paths L', xs)
        rows = [dict(model=m, **{x: r[m][b]['pairing_error'] for b, x in zip(banks, xs)}) for m in ms]
        put_table(f'E7_pairing_snr{snr}', f'E7 pairing-error rate (exactly one of AoA/AoD correct), SNR {snr} dB',
                  md_table(rows, ['model'] + xs))
        plot_sweep(r, ms, banks, xs, 'number of paths L', f'E7_pathcount_snr{snr}.png'); out[snr] = r
    return dict(results=out)


def e8():
    seps = [30, 20, 10, 5, 3, 2, 1]
    banks = [f'sep_{s}deg_snr15' for s in seps]
    r, ms = sweep('E8', 'E8 angular separation (L=2, SNR 15 dB; both AoA and AoD offset)', banks, 'separation (deg)', seps)
    plot_sweep(r, ms, banks, seps, 'separation (deg)', 'E8_separation.png', key='pd_source')
    return dict(results=r)


def tails():
    snrs = [-25, -20, 30, 35]
    banks = [f'tail_snr{s}' for s in snrs]
    r, ms = sweep('SNR_tails', 'SNR extrapolation tails (L=3, outside the -15..24 dB training range)', banks, 'SNR (dB)', snrs)
    return dict(results=r)


R_E4 = run_exp('E4_phase_error_sweep', e4, needs=['BANKS_build'])
R_E5 = run_exp('E5_gain_error_sweep', e5, needs=['BANKS_build'])
R_E6 = run_exp('E6_ood_phase_error', e6, needs=['BANKS_build'])
R_E7 = run_exp('E7_path_count_sweep', e7, needs=['BANKS_build'])
R_E8 = run_exp('E8_angular_separation', e8, needs=['BANKS_build'])
R_TAILS = run_exp('SNR_tails_extrapolation', tails, needs=['BANKS_build'])

In [ ]:
# ---------------------------------------------------------------- 18. E10 -- nuisance-path stress test (Notebook-4 protocol)
def e10():
    models = [m for m in ['Teacher', 'Student', 'FNO', 'DFT-SIC', MAIN, 'Abl1_no_IABC'] if m in BASE or m == 'DFT-SIC' or trained(m)]
    rows, res = [], {}
    for snr in CFG['robust_snrs']:
        for pw in [-20, -10, 0]:
            b = load_bank(f'nuis_p{pw}_snr{snr}')
            for m in models:
                pr = predict(m, f'nuis_p{pw}_snr{snr}')
                principal = metrics(pr, b, sub=slice(0, 3)); nuis = metrics(pr, b, sub=slice(3, 4))
                res[f'{m}|{snr}|{pw}'] = dict(principal=principal, nuisance=nuis)
                rows.append(dict(model=m, snr=snr, nuisance_db=pw, pd_principal=principal['pd_paper'],
                                 pd_principal_strict=principal['pd_strict'], pd_nuisance=nuis['pd_paper']))
    put_table('E10', 'E10 — nuisance-path stress test (3 principal paths + 1 interferer; paired scenes)',
              md_table(rows, ['model', 'snr', 'nuisance_db', 'pd_principal', 'pd_principal_strict', 'pd_nuisance']))
    return dict(results=res, rows=rows)


R_E10 = run_exp('E10_nuisance_path', e10, needs=['BANKS_build'])

In [ ]:
# ---------------------------------------------------------------- 19. Controlled comparisons: E1 / E2 / Abl-1 / Abl-3
def controlled():
    names = available_iabr([MAIN, 'E1_capacity_control', 'E2_dense_frontend', 'Abl1_no_IABC', 'Abl3_no_SE'])
    conds = ['E3_frozen', 'phase_d5_snr15', 'gain_g2_snr15', 'gain_g4_snr15', 'oodphase_d10_snr15']
    rows, res = [], {}
    for n in names:
        row = dict(model=n, params=int(get_iabr(n).count_params()))
        for c in conds:
            b = load_bank(c)
            r = by_snr(predict(n, c), b)['mean_over_snr'] if c == 'E3_frozen' else metrics(predict(n, c), b)
            res[f'{n}|{c}'] = r
            row[c] = r['pd_paper']
        rows.append(row)
    put_table('Controls', 'E1 / E2 / Abl-1 / Abl-3 — same data, one component changed (paper-style Pd; E3 = mean over SNR)',
              md_table(rows, ['model', 'params'] + conds, fmt={'params': '{:,}'}))
    return dict(results=res, rows=rows)


R_CTRL = run_exp('Controls_E1_E2_Abl1_Abl3', controlled, needs=['BANKS_build'])

In [ ]:
# ---------------------------------------------------------------- 20. E11 multi-seed variance + Abl-6 loss-weight sweep
def e11():
    seeds = available_iabr([f'IABR_s{s}' for s in CFG['seeds']])
    conds = ['E3_frozen', 'phase_d5_snr15', 'gain_g2_snr15']
    vals = {c: [] for c in conds}
    for n in seeds:
        for c in conds:
            b = load_bank(c)
            r = by_snr(predict(n, c), b)['mean_over_snr'] if c == 'E3_frozen' else metrics(predict(n, c), b)
            vals[c].append(r['pd_paper'])
    rows = [dict(condition=c, n_seeds=len(v), mean=float(np.mean(v)) if v else np.nan,
                 std=float(np.std(v, ddof=1)) if len(v) > 1 else np.nan, values=str([round(x, 4) for x in v])) for c, v in vals.items()]
    put_table('E11', 'E11 — multi-seed variance of IABR-Net (paper-style Pd)', md_table(rows, ['condition', 'n_seeds', 'mean', 'std', 'values']))
    return dict(rows=rows)


def abl6():
    names = available_iabr([n for n in VARIANTS if n.startswith('Abl6')])
    fb = load_bank('E3_frozen')
    k = CFG['sweep_eval_n_per_snr']
    snr_vals = fb['snr'].astype(int)
    idx = np.concatenate([np.where(snr_vals == s)[0][:k] for s in sorted(set(snr_vals.tolist()))])
    rows = []
    for n in names:
        _, _, lam, _ = VARIANTS[n]
        e3r = metrics(predict(n, 'E3_frozen'), fb, idx)
        p5 = metrics(predict(n, 'phase_d5_snr15'), load_bank('phase_d5_snr15'))
        g2 = metrics(predict(n, 'gain_g2_snr15'), load_bank('gain_g2_snr15'))
        rows.append(dict(run=n, lambda3=lam['pair'], lambda4=lam['snr'], E3_subset_pd=e3r['pd_paper'],
                         phase5_pd=p5['pd_paper'], gain2_pd=g2['pd_paper']))
    put_table('Abl6', f'Abl-6 — loss-weight sensitivity ({CFG["sweep_steps"]} steps per run; E3 subset = {k}/SNR)',
              md_table(rows, ['run', 'lambda3', 'lambda4', 'E3_subset_pd', 'phase5_pd', 'gain2_pd']))
    return dict(rows=rows)


R_E11 = run_exp('E11_multi_seed', e11, needs=['BANKS_build'])
R_ABL6 = run_exp('Abl6_loss_weight_sweep', abl6, needs=['BANKS_build'])

In [ ]:
# ---------------------------------------------------------------- 21. E9 -- latency, FLOPs, memory, model size (every model)
def time_it(fn, x, iters):
    for _ in range(3):
        _ = fn(x)
    ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); r = fn(x)
        (r[0] if isinstance(r, (tuple, list)) else r).numpy()
        ts.append((time.perf_counter() - t0) * 1000)
    return float(np.percentile(ts, 50)), float(np.percentile(ts, 99)), float(np.mean(ts))


def gpu_peak_mb(fn, x):
    if not GPUS:
        return None
    try:
        tf.config.experimental.reset_memory_stats('GPU:0')
        r = fn(x); (r[0] if isinstance(r, (tuple, list)) else r).numpy()
        return tf.config.experimental.get_memory_info('GPU:0')['peak'] / 2 ** 20
    except Exception:
        return None


def e9():
    rows, res = [], {}
    bank = load_bank('E3_frozen')
    ns = min(CFG['latency_e2e_samples'], len(bank['L']))
    specs = {}
    for name, m in BASE.items():
        specs[name] = dict(fn=tf.function(lambda x, m=m: m(x, training=False)), shape=(64, 64, 2), params=int(m.count_params()),
                           wfile={'Teacher': TEACHER_W, 'Student': STUDENT_W, 'FNO': FNO_W}[name],
                           flops_extra=fno_fft_flops() if name == 'FNO' else 0)
    if trained(MAIN):
        mi = get_iabr(MAIN)
        specs['IABR-Net'] = dict(fn=tf.function(iabr_infer_fn(mi)), shape=(Q, P, 2), params=int(mi.count_params()),
                                 wfile=os.path.join(DIRS['checkpoints'], MAIN, 'final.weights.h5'),
                                 flops_extra=complex_einsum_flops_iabr())
    for name, sp in specs.items():
        prof, counted, _ = profiler_flops(sp['fn'], [tf.TensorSpec([1] + list(sp['shape']), tf.float32)])
        lat = {}
        for bs in CFG['latency_batches']:
            x = tf.random.normal([bs] + list(sp['shape']))
            try:
                p50, p99, mean = time_it(sp['fn'], x, CFG['latency_iters'])
                lat[bs] = dict(p50_ms=p50, p99_ms=p99, mean_ms=mean, throughput_per_s=bs / (p50 / 1000))
            except Exception as e:
                lat[bs] = dict(error=str(e))
        peak = gpu_peak_mb(sp['fn'], tf.random.normal([32] + list(sp['shape'])))
        run_one = (lambda one: predict_heatmap(BASE[name], one)) if name in BASE else (lambda one: predict_iabr(get_iabr(MAIN), one))
        run_one({'Y16': bank['Y16'][:1], 'L': bank['L'][:1]})                         # warm-up / trace batch-1 graph
        e2e = []
        for i in range(ns):
            one = {'Y16': bank['Y16'][i:i + 1], 'L': bank['L'][i:i + 1]}
            t0 = time.perf_counter(); run_one(one); e2e.append((time.perf_counter() - t0) * 1000)
        res[name] = dict(params=sp['params'], flops_profiler=prof, flops_supplement=sp['flops_extra'],
                         flops_total=prof + sp['flops_extra'], counted_ops=counted, latency_nn=lat,
                         gpu_peak_mb_batch32=peak, weights_file_mb=os.path.getsize(sp['wfile']) / 2 ** 20,
                         param_memory_mb=sp['params'] * 4 / 2 ** 20,
                         e2e_p50_ms=float(np.percentile(e2e, 50)), e2e_p99_ms=float(np.percentile(e2e, 99)))
    Yc = bank['Y16'][..., 0] + 1j * bank['Y16'][..., 1]
    e2e = []
    for i in range(ns):
        t0 = time.perf_counter(); classical_dft_sic(Yc[i], int(bank['L'][i]), CFG['dft_ndft']); e2e.append((time.perf_counter() - t0) * 1000)
    res['DFT-SIC'] = dict(params=0, flops_total=None, e2e_p50_ms=float(np.percentile(e2e, 50)), e2e_p99_ms=float(np.percentile(e2e, 99)))
    e3r = json.load(open(os.path.join(DIRS['results'], 'E3_in_distribution_snr_sweep.json'))) if os.path.exists(
        os.path.join(DIRS['results'], 'E3_in_distribution_snr_sweep.json')) else None
    for name, r in res.items():
        key = MAIN if name == 'IABR-Net' else name
        acc = (e3r or {}).get('results', {}).get(key, {}).get('mean_over_snr', {}) if e3r else {}
        b1 = r.get('latency_nn', {}).get(1, r.get('latency_nn', {}).get('1', {})) if 'latency_nn' in r else {}
        rows.append(dict(model=name, params=r['params'], GFLOPs=(r['flops_total'] / 1e9) if r.get('flops_total') else np.nan,
                         weights_MB=r.get('weights_file_mb', np.nan), gpu_peak_MB=r.get('gpu_peak_mb_batch32') or np.nan,
                         nn_ms_b1=b1.get('p50_ms', np.nan) if isinstance(b1, dict) else np.nan,
                         e2e_ms=r['e2e_p50_ms'], e2e_p99_ms=r['e2e_p99_ms'],
                         mean_pd=acc.get('pd_paper', np.nan), mean_rmse=acc.get('rmse', np.nan)))
    put_table('E9', f'E9 — efficiency ({DEVICE}; NN latency = p50 at batch 1; e2e = NN + peak decoding per sample, p50)',
              md_table(rows, ['model', 'params', 'GFLOPs', 'weights_MB', 'gpu_peak_MB', 'nn_ms_b1', 'e2e_ms', 'e2e_p99_ms', 'mean_pd', 'mean_rmse'],
                       fmt={'params': '{:,}', 'GFLOPs': '{:.3f}', 'weights_MB': '{:.2f}', 'gpu_peak_MB': '{:.0f}',
                            'nn_ms_b1': '{:.3f}', 'e2e_ms': '{:.2f}', 'e2e_p99_ms': '{:.2f}'}))
    return dict(results=res, device=DEVICE, gpu=str(GPUS))


R_E9 = run_exp('E9_latency_flops_memory', e9, needs=['E0a_phase0_unit_tests'])

In [ ]:
# ---------------------------------------------------------------- 22. Final report: outputs/RESULTS.md + summary
def write_report():
    lines = ['# IABR-Net test suite — results', '',
             f'- Generated: {datetime.datetime.now().isoformat(timespec="seconds")}',
             f'- Device: {DEVICE} {GPUS} | TensorFlow {tf.__version__}',
             f'- SMOKE_TEST: **{SMOKE_TEST}**' + ('  ⚠️ tiny settings, numbers are NOT meaningful' if SMOKE_TEST else ''),
             f'- Total wall-clock this session: {(time.time() - T_START) / 3600:.2f} h', '',
             '## Status of every experiment', '', '| experiment | status |', '|---|---|']
    lines += [f'| {k} | {v} |' for k, v in STATUS.items()]
    lines += ['', 'All numbers use the original project evaluator (Hungarian matching, 1° threshold). `pd_paper` is directly '
              'comparable to every earlier number in this project; `pd_strict` counts dropped samples as misses.', '']
    if R_E0A:
        lines += ['## E0a — Phase-0 unit tests', '', '| check | value | pass |', '|---|---|---|']
        lines += [f'| {k} | {v} | {R_E0A["passed"].get(k)} |' for k, v in R_E0A['checks'].items()]
        lines.append('')
    if R_V0:
        lines += ['## V0 — teacher reproduces the registry', '',
                  f'Teacher mean Pd on the frozen bank = **{R_V0["teacher_mean_pd"]:.10f}** (registry {R_V0["registry"]}) → '
                  f'{"PASS" if R_V0["passed"] else "FAIL"}', '']
    if R_E0B:
        lines += ['## E0b — IABR-Net measured size', '',
                  f'- Parameters: **{R_E0B["params_total"]:,}** (trainable {R_E0B["params_trainable"]:,}); report estimate 193,104',
                  f'- FLOPs (batch 1, profiler + complex-einsum supplement): **{R_E0B["flops_total"] / 1e6:.2f} M**; report estimate 95.3 M',
                  f'- Parameter memory: {R_E0B["param_memory_MB"]:.2f} MB', '']
    order = ['V1', 'E3', 'Abl2', 'Controls', 'E4', 'E5', 'E6', 'E7', 'E8', 'SNR_tails', 'E10', 'E11', 'Abl6', 'E9']
    tfiles = sorted(os.listdir(DIRS['tables']),
                    key=lambda f: (next((i for i, o in enumerate(order) if f.startswith(o)), 99), f))
    for tf_name in tfiles:                     # read from disk so tables from cached (earlier-session) experiments are included
        with open(os.path.join(DIRS['tables'], tf_name), encoding='utf-8') as f:
            lines += [f.read().replace('### ', '## ', 1), '']
    lines += ['## Figures', ''] + [f'- `figures/{f}`' for f in sorted(os.listdir(DIRS['figures']))]
    fails = {k: v for k, v in STATUS.items() if not str(v).startswith('done')}
    lines += ['', '## Not completed', ''] + ([f'- {k}: {v}' for k, v in fails.items()] or ['- none'])
    with open(os.path.join(OUT, 'RESULTS.md'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    return fails


FAILS = write_report()
log('=' * 70)
log('RESULTS written to', os.path.relpath(os.path.join(OUT, 'RESULTS.md'), ROOT))
for k, v in STATUS.items():
    log(f'  {k:<40} {v}')
log('not completed:', len(FAILS))
log(f'total session time {(time.time() - T_START) / 3600:.2f} h')